# UPI Fraud Rings — Teammate Branch (`codex/target-070`) Full A100 Run
Runs `xgb_a100.yaml` + `catboost_a100.yaml`, generates test predictions for both, blends them, and saves all artifacts back to Google Drive.

In [ ]:
!nvidia-smi -L
!nproc
!free -h

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/zahy294/upi-fraud-ml.git'
BRANCH = 'codex/target-070'
!rm -rf /content/upi-fraud-ml
!git clone --branch {BRANCH} --single-branch {REPO_URL} /content/upi-fraud-ml
%cd /content/upi-fraud-ml
!git rev-parse HEAD
!python -c "from pathlib import Path; p=Path('src/upi/model.py'); s=p.read_text(); p.write_text(s.replace('return XGBClassifier(**defaults)', 'if n_estimators is not None: defaults[\"early_stopping_rounds\"] = None\n        return XGBClassifier(**defaults)'))"
!python -m pip install -q pyarrow xgboost catboost lightgbm scikit-learn PyYAML psutil

In [ ]:
# Automatically finds either upi-fraud-rings-gradient-wars.zip OR the raw CSVs on your Google Drive
import os, glob, shutil, zipfile
os.makedirs('data/raw', exist_ok=True)

zip_matches = glob.glob('/content/drive/MyDrive/**/upi-fraud-rings-gradient-wars.zip', recursive=True)
csv_matches = glob.glob('/content/drive/MyDrive/**/train.csv', recursive=True)

if zip_matches:
    print('Found zip on Drive:', zip_matches[0])
    with zipfile.ZipFile(zip_matches[0], 'r') as z:
        z.extractall('/tmp/upi_unzip')
    for f in glob.glob('/tmp/upi_unzip/**/*.csv', recursive=True):
        shutil.copy2(f, 'data/raw/' + os.path.basename(f))
elif csv_matches:
    src_dir = os.path.dirname(csv_matches[0])
    print('Found CSV folder on Drive:', src_dir)
    for f in glob.glob(src_dir + '/*.csv'):
        shutil.copy2(f, 'data/raw/' + os.path.basename(f))
else:
    raise FileNotFoundError('Please upload upi-fraud-rings-gradient-wars.zip anywhere in your Google Drive (MyDrive)!')

!ls -lh data/raw/
!PYTHONPATH=src python run.py audit --config configs/xgb_a100.yaml
!PYTHONPATH=src python -m unittest discover -s tests -v

In [ ]:
# Build features + Train & Predict XGBoost A100 + CatBoost A100 + Blend
import subprocess, json
from pathlib import Path

!PYTHONPATH=src python run.py features --config configs/xgb_a100.yaml

xgb_run = subprocess.check_output(['python', 'run.py', 'train', '--config', 'configs/xgb_a100.yaml'], text=True).strip().splitlines()[-1]
print('XGBoost validation run:', xgb_run)
xgb_manifest = json.loads((Path(xgb_run) / 'manifest.json').read_text())
print('XGBoost Sep Assessment Macro-F1:', xgb_manifest['reports']['assessment']['macro_f1'])
print('XGBoost Full Sep Macro-F1:', xgb_manifest['reports']['full_september']['macro_f1'])
!PYTHONPATH=src python run.py predict --config configs/xgb_a100.yaml --run-dir {xgb_run}

cat_run = subprocess.check_output(['python', 'run.py', 'train', '--config', 'configs/catboost_a100.yaml'], text=True).strip().splitlines()[-1]
print('CatBoost validation run:', cat_run)
cat_manifest = json.loads((Path(cat_run) / 'manifest.json').read_text())
print('CatBoost Sep Assessment Macro-F1:', cat_manifest['reports']['assessment']['macro_f1'])
print('CatBoost Full Sep Macro-F1:', cat_manifest['reports']['full_september']['macro_f1'])
!PYTHONPATH=src python run.py predict --config configs/catboost_a100.yaml --run-dir {cat_run}

blend_run = subprocess.check_output(['python', 'run.py', 'blend', '--config', 'configs/xgb_a100.yaml', '--run-dirs', xgb_run, cat_run], text=True).strip().splitlines()[-1]
blend_manifest = json.loads((Path(blend_run) / 'manifest.json').read_text())
print('Blend Sep Assessment Macro-F1:', blend_manifest['reports']['assessment']['macro_f1'])
print('Blend Full Sep Macro-F1:', blend_manifest['reports']['full_september']['macro_f1'])

!mkdir -p /content/drive/MyDrive/upi-fraud/runs /content/drive/MyDrive/upi-fraud/submissions
!cp -r runs/* /content/drive/MyDrive/upi-fraud/runs/
!cp -r submissions/* /content/drive/MyDrive/upi-fraud/submissions/
print('All runs and submissions saved to /content/drive/MyDrive/upi-fraud/')